## Import des librairies

In [1]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, 
    f1_score, 
    precision_score, 
    recall_score, confusion_matrix, 
    ConfusionMatrixDisplay, 
    classification_report, 
    roc_curve,
    RocCurveDisplay,
    PrecisionRecallDisplay
)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.svm import LinearSVC

In [2]:
X_train = pd.read_csv('X_train_clean.csv')
X_test = pd.read_csv('X_test_clean.csv')
y_train = pd.read_csv('y_train.csv')
y_test = pd.read_csv('y_test.csv')

In [3]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(242, 17)
(61, 17)
(242, 1)
(61, 1)


## Hyperparameters KNN

In [4]:
n_neighbors = 5
weights = 'uniform'
algorithm = 'auto'
leaf_size = 30
p = 2
metric = 'minkowski'

model_knn = KNeighborsClassifier(
    n_neighbors=n_neighbors,
    weights=weights,
    algorithm=algorithm,
    leaf_size=leaf_size,    
    p=p,
    metric=metric
)

model_knn

y_train.values.ravel()


model_knn.fit(X_train, y_train.values.ravel())

KNeighborsClassifier()

In [5]:
y_pred = model_knn.predict(X_test.values)

C:\Users\anasa\AppData\Roaming\jupyterlab-desktop\jlab_server\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(


In [6]:
# comparer les valeurs de prédiction de X_test avec y_test
y_test

,target
0,0
1,1
2,0
3,0
4,0
...,...
56,1
57,0
58,1
59,0


## Etudier les metrics

In [7]:
# Exactitude : Accuracy
acc_score = accuracy_score(y_pred, y_test)
print(acc_score)

0.8360655737704918


## GridSearchCV (Recherche du meilleur modèle)

In [8]:
# Recherche des meilleurs paramètres
hyperparametres = {
    'n_neighbors' : list(range(3, 20)),
    'weights' : ['uniform', 'distance'],
    'algorithm' : ['ball_tree', 'kd_tree'],
    'p' : list(range(2, 5))    
}

gscv_knn = GridSearchCV(
    estimator=model_knn, 
    param_grid=hyperparametres,
    cv=5,
    scoring= 'f1',
    verbose= 4
)

# lancer le modèle
gscv_knn.fit(X_train, y_train.values.ravel())

Fitting 5 folds for each of 204 candidates, totalling 1020 fits
[CV 1/5] END algorithm=ball_tree, n_neighbors=3, p=2, weights=uniform;, score=0.727 total time=   0.0s
[CV 2/5] END algorithm=ball_tree, n_neighbors=3, p=2, weights=uniform;, score=0.773 total time=   0.0s
[CV 3/5] END algorithm=ball_tree, n_neighbors=3, p=2, weights=uniform;, score=0.829 total time=   0.0s
[CV 4/5] END algorithm=ball_tree, n_neighbors=3, p=2, weights=uniform;, score=0.844 total time=   0.0s
[CV 5/5] END algorithm=ball_tree, n_neighbors=3, p=2, weights=uniform;, score=0.773 total time=   0.0s
[CV 1/5] END algorithm=ball_tree, n_neighbors=3, p=2, weights=distance;, score=0.698 total time=   0.0s
[CV 2/5] END algorithm=ball_tree, n_neighbors=3, p=2, weights=distance;, score=0.708 total time=   0.0s
[CV 3/5] END algorithm=ball_tree, n_neighbors=3, p=2, weights=distance;, score=0.769 total time=   0.0s
[CV 4/5] END algorithm=ball_tree, n_neighbors=3, p=2, weights=distance;, score=0.818 total time=   0.0s
[CV 5

GridSearchCV(cv=5, estimator=KNeighborsClassifier(),
             param_grid={'algorithm': ['ball_tree', 'kd_tree'],
                         'n_neighbors': [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,
                                         14, 15, 16, 17, 18, 19],
                         'p': [2, 3, 4], 'weights': ['uniform', 'distance']},
             scoring='f1', verbose=4)

In [9]:
# Chercher les meilleurs hyperparamètres correspondants
best_knn = gscv_knn.best_estimator_
print(best_knn)

KNeighborsClassifier(algorithm='ball_tree', n_neighbors=7, p=4)


In [10]:
best_knn_hyperparameters = gscv_knn.best_params_
print(best_knn_hyperparameters)

{'algorithm': 'ball_tree', 'n_neighbors': 7, 'p': 4, 'weights': 'uniform'}


In [11]:
# Il y a plus d'exemple dans le train que dans le test
y_pred = best_knn.predict(X_test)

In [12]:
y_pred.shape

(61,)

In [13]:
y_test.shape

(61, 1)

In [14]:
print("Accuracy :", accuracy_score(y_pred, y_test.values))
print("Precision :", precision_score(y_pred, y_test.values))
print("Recall :", recall_score(y_pred, y_test.values))
print("F1 :", f1_score(y_pred, y_test.values))

Accuracy : 0.8524590163934426
Precision : 0.8571428571428571
Recall : 0.8275862068965517
F1 : 0.8421052631578947


In [15]:
print(classification_report(y_pred, y_test))

              precision    recall  f1-score   support

           0       0.85      0.88      0.86        32
           1       0.86      0.83      0.84        29

    accuracy                           0.85        61
   macro avg       0.85      0.85      0.85        61
weighted avg       0.85      0.85      0.85        61



## Classification avec LinearSVC

In [16]:
hyperparametres = [
    {
         "penalty" : ["l1", "l2"],
         "loss" : ["squared_hinge"],
         "dual" : [False],
         "C" : [0.1, 1, 2, 3, 10],
         
    }
]

gscv_knn = GridSearchCV(
    estimator= LinearSVC(),
    param_grid= hyperparametres,
    cv = 5,
    scoring= "f1",
    verbose= 4
)

gscv_knn.fit(X_train, y_train.values.ravel())

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV 1/5] END C=0.1, dual=False, loss=squared_hinge, penalty=l1;, score=0.714 total time=   0.0s
[CV 2/5] END C=0.1, dual=False, loss=squared_hinge, penalty=l1;, score=0.864 total time=   0.0s
[CV 3/5] END C=0.1, dual=False, loss=squared_hinge, penalty=l1;, score=0.829 total time=   0.0s
[CV 4/5] END C=0.1, dual=False, loss=squared_hinge, penalty=l1;, score=0.884 total time=   0.0s
[CV 5/5] END C=0.1, dual=False, loss=squared_hinge, penalty=l1;, score=0.727 total time=   0.0s
[CV 1/5] END C=0.1, dual=False, loss=squared_hinge, penalty=l2;, score=0.683 total time=   0.0s
[CV 2/5] END C=0.1, dual=False, loss=squared_hinge, penalty=l2;, score=0.837 total time=   0.0s
[CV 3/5] END C=0.1, dual=False, loss=squared_hinge, penalty=l2;, score=0.780 total time=   0.0s
[CV 4/5] END C=0.1, dual=False, loss=squared_hinge, penalty=l2;, score=0.837 total time=   0.0s
[CV 5/5] END C=0.1, dual=False, loss=squared_hinge, penalty=l2;, score=0.74

GridSearchCV(cv=5, estimator=LinearSVC(),
             param_grid=[{'C': [0.1, 1, 2, 3, 10], 'dual': [False],
                          'loss': ['squared_hinge'], 'penalty': ['l1', 'l2']}],
             scoring='f1', verbose=4)